# DPO 전처리 및 데이터셋 구축

## 전체 파이프라인

| 단계 | 작업 | 출력 |
|------|------|------|
| [1단계] | LLM으로 새 리뷰 생성 | `DPO_aug_data.csv` |
| [2단계] | LLM 출력 태그 제거 + 리스트 파싱 | `DPO_aug_data_pre.csv` |
| [3단계] | 수동 검수 + flatten + 중복 제거 | `DPO_aug_data_cleaned_duplicate_drop.csv` |
| [4단계] | 난독화 크롤링 (3회 실행) | `DPO_aug_pre_final1~3.csv` |
| [5단계] | SFT 모델 추론 → Hard Negatives → DPO 데이터셋 구성 | `DPO_dataset_final.csv` |

In [ ]:
!pip install accelerate==1.2.1 bitsandbytes==0.45.1 peft==0.14.0 transformers==4.48.2 trl==0.14.0 vllm==0.7.1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import re
import ast
import pandas as pd
from tqdm import tqdm

# ── 경로 설정 ─────────────────────────────────────────────────────────────
BASE_DIR = '/content/drive/MyDrive/Colab Notebooks/open'

# [1단계] LLM 리뷰 생성
TRAIN_CSV          = os.path.join(BASE_DIR, 'train.csv')
AUG_DATA_CSV       = os.path.join(BASE_DIR, 'DPO_aug_data.csv')
AUG_DATA_CKPT_CSV  = os.path.join(BASE_DIR, 'DPO_aug_data_checkpoint.csv')

# [2단계] LLM 출력 클렌징
AUG_DATA_PRE_CSV   = os.path.join(BASE_DIR, 'DPO_aug_data_pre.csv')

# [3단계] 수동 검수 + flatten
AUG_CLEANED_CSV    = os.path.join(BASE_DIR, 'DPO_aug_data_cleaned_duplicate_drop.csv')

# [4단계] 난독화 크롤링 (3회 실행)
CRAWL_CKPT_CSV     = os.path.join(BASE_DIR, 'DPO_crawl_checkpoint.csv')
FINAL_CSV_1        = os.path.join(BASE_DIR, 'DPO_aug_pre_final1.csv')
FINAL_CSV_2        = os.path.join(BASE_DIR, 'DPO_aug_pre_final2.csv')
FINAL_CSV_3        = os.path.join(BASE_DIR, 'DPO_aug_pre_final3.csv')

# [5단계] DPO 데이터셋 구축
AUG_CSV            = os.path.join(BASE_DIR, 'augmented_texts.csv')      # sft_preprocessing 결과
AUG_CSV_50         = os.path.join(BASE_DIR, 'augmented_texts50.csv')
AUG_CSV_100        = os.path.join(BASE_DIR, 'augmented_texts100.csv')
OUTPUT_SFT_CSV     = os.path.join(BASE_DIR, 'DPO_train_data.csv')
OUTPUT_AUG_CSV     = os.path.join(BASE_DIR, 'DPO_aug_data_inference.csv')
OUTPUT_FINAL_CSV   = os.path.join(BASE_DIR, 'DPO_dataset_final.csv')

## [1단계] LLM 기반 리뷰 데이터 증강

- **목적**: 대회 제공 `train.csv`의 원문 리뷰를 예시로 활용해 LLM으로 새로운 리뷰 데이터를 생성
- **방법**: 원문 리뷰 3개를 묶어 프롬프트 예시로 제공 → Llama 3.1 Korean 모델이 유사한 스타일의 새 리뷰 3개 생성
- **출력**: `DPO_aug_data.csv` → [2단계] 클렌징에 사용

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

AUG_MODEL_NAME = "MLP-KTLim/llama-3-Korean-Bllossom-8B"

vllm_aug_model = LLM(
    model=AUG_MODEL_NAME,
    tokenizer=AUG_MODEL_NAME,
    gpu_memory_utilization=0.9,
    max_model_len=4096
)

aug_tokenizer = AutoTokenizer.from_pretrained(AUG_MODEL_NAME)

# 종료 토큰 설정
eos_token_id = aug_tokenizer.eos_token_id
eot_token_id = aug_tokenizer.convert_tokens_to_ids("<|eot_id|>")

aug_sampling_params = SamplingParams(
    stop_token_ids=[eos_token_id, eot_token_id],
    max_tokens=2048,
    temperature=0.6
)

In [ ]:
train = pd.read_csv(TRAIN_CSV, encoding='utf-8-sig')
print(f'train 데이터 크기: {len(train)}')
train['output'].head(5)

In [ ]:
AUG_SYSTEM_PROMPT = """You are a helpful AI assistant. Please answer the user's questions kindly.
당신은 유능한 AI 어시스턴트 입니다. 사용자의 질문에 대해 친절하게 답변해주세요."""

USER_INSTRUCTION = """
데이터 증강을 하려고 하는데 아래 Response의 예시 리뷰를 참고해서 실제 사람들이 남길만한 리뷰 데이터 3개를 새롭게 만들어줘.
평점 및 따옴표는 빼고 한글로만 리뷰를 남겨줘.
반드시 텍스트는 아래 형식처럼 하나의 리스트로 만들어야만 해!
<Response>
['{}', '{}', '{}']
</Response>
"""

In [ ]:
batch_size  = 128
num_samples = len(train['output']) - 1
correct_data = []

for batch_start in tqdm(range(0, num_samples, batch_size), desc="Augmenting reviews"):
    batch_end     = min(batch_start + batch_size, num_samples)
    batch_prompts = []

    # 원문 리뷰 3개씩 묶어서 프롬프트 생성
    for i in range(batch_start, batch_end, 3):
        if i + 2 >= num_samples:  # 마지막 배치에서 index 초과 방지
            break
        try:
            messages = [
                {"role": "system", "content": AUG_SYSTEM_PROMPT},
                {"role": "user",   "content": USER_INSTRUCTION.format(
                    train['output'][i],
                    train['output'][i + 1],
                    train['output'][i + 2]
                )}
            ]
            prompt = aug_tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            batch_prompts.append(prompt)
        except Exception as e:
            print(f"[ERROR] 프롬프트 생성 실패 idx={i} → {e}")
            continue

    # vLLM 배치 추론
    if batch_prompts:
        try:
            batch_outputs = vllm_aug_model.generate(batch_prompts, aug_sampling_params)
            for response in batch_outputs:
                correct_data.append(response.outputs[0].text)
        except Exception as e:
            print(f"[ERROR] 배치 추론 실패 batch_start={batch_start} → {e}")
            continue

    # 100배치마다 중간 저장 (크래시 시 데이터 손실 방지)
    if (batch_start // batch_size + 1) % 100 == 0:
        pd.DataFrame(correct_data, columns=["aug_data"]).to_csv(
            AUG_DATA_CKPT_CSV, index=False, encoding='utf-8-sig'
        )
        print(f"[CHECKPOINT] {len(correct_data)}개 저장 완료")

print(f"\n증강 완료: 총 {len(correct_data)}개 생성")

In [ ]:
df_aug = pd.DataFrame(correct_data, columns=["aug_data"])
df_aug.to_csv(AUG_DATA_CSV, index=False, encoding='utf-8-sig')
print(f"저장 완료 → {AUG_DATA_CSV}")
df_aug.head()

## [2단계] LLM 출력 클렌징

- **목적**: [1단계]에서 생성된 LLM 출력 텍스트의 잔여 태그 제거 및 리스트 형식 파싱
- **처리**
  - 1단계: `[Response]`, `<Response>`, `<리뷰>` 등 잔여 태그 제거
  - 2단계: 리스트 형식 `[...]` 부분만 추출 후 실제 파이썬 리스트로 변환
- **출력**: `DPO_aug_data_pre.csv` → [3단계] 수동 검수에 사용

In [ ]:
aug_data = pd.read_csv(AUG_DATA_CSV, encoding='utf-8-sig')
print(f'데이터 크기: {len(aug_data)}')
aug_data.head(5)

In [ ]:
# 1단계: LLM 출력에 남아있는 잔여 태그 제거
PATTERNS_TO_REMOVE = [
    r"\[Response\]", r"\<Response\>", r"\(Response\)",
    r"\[/?Response\]", r"\</?Response\>", r"\(/?Response\)",
    r"\<리뷰\>", r"\</?리뷰\>",
    r"\<Responses\>", r"\</?Responses\>"
]

def remove_patterns(text):
    if not isinstance(text, str):
        return text
    for pattern in PATTERNS_TO_REMOVE:
        text = re.sub(pattern, "", text).strip()
    return text

aug_data1 = aug_data.copy()
aug_data1['aug_data'] = aug_data['aug_data'].apply(remove_patterns)
aug_data1.head(5)

In [ ]:
# 2단계: 리스트 형식 [...] 부분만 추출 후 파이썬 리스트로 변환
# 변환 실패 시 원본 텍스트 유지
def extract_list_content(text):
    if not isinstance(text, str):
        return text

    matches = re.findall(r"\[.*?\]", text, re.DOTALL)

    if matches:
        try:
            return ast.literal_eval(matches[0])  # 문자열 → 실제 리스트 변환
        except (SyntaxError, ValueError):
            return text  # 변환 실패 시 원본 유지

    return text  # 리스트 패턴 없으면 원본 유지

aug_data2 = aug_data1.copy()
aug_data2['aug_data'] = aug_data1['aug_data'].apply(extract_list_content)
aug_data2.head(5)

In [ ]:
aug_data2.to_csv(AUG_DATA_PRE_CSV, index=False, encoding='utf-8-sig')
print(f'저장 완료 → {AUG_DATA_PRE_CSV}')
aug_data2.info()

## [3단계] 수동 검수 + flatten + 중복 제거

> ⚠️ **수동 작업 필요**: 아래 셀 실행 전 `DPO_aug_data_pre.csv`를 열어 각 행의 `check` 컬럼에 `O` / `X`를 직접 입력해주세요.
> - `O`: 품질 양호 → 이후 단계에서 사용
> - `X`: 품질 불량 (이상한 리뷰, 파싱 실패 등) → 제외
>
> 검수 완료 후 파일 저장 → 아래 셀 실행

- **처리**
  - 1단계: `check == 'O'` 행만 필터링
  - 2단계: 대괄호 제거 후 작은따옴표 기준으로 리뷰 개별 추출
  - 3단계: 중첩 리스트 → 1차원 flatten + 중복 제거
- **출력**: `DPO_aug_data_cleaned_duplicate_drop.csv` → [4단계] 난독화 크롤링에 사용

In [ ]:
aug_data_pre = pd.read_csv(AUG_DATA_PRE_CSV, encoding='utf-8-sig')
print(f'전체 데이터 크기: {len(aug_data_pre)}')
aug_data_pre.head()

In [ ]:
# 1단계: 수동 검수 통과 데이터만 필터링 (check == 'O')
aug_data_checked = aug_data_pre[aug_data_pre['check'] == 'O'].reset_index(drop=True)
print(f'검수 통과 데이터: {len(aug_data_checked)}개')

In [ ]:
# 2단계: 대괄호 제거 후 작은따옴표 기준으로 리뷰 개별 추출
def convert_to_list(text):
    if isinstance(text, str):
        text = re.sub(r"[\[\]]", "", text)       # 대괄호 제거
        return re.findall(r"'(.*?)'", text)         # 작은따옴표 안의 내용 추출
    return text

aug_data_checked['aug_data_cleaned'] = aug_data_checked['aug_data'].apply(convert_to_list)
aug_data_checked['aug_data_cleaned'].head()

In [ ]:
# 3단계: 중첩 리스트 → 1차원 flatten (문장 단위로 정리)
aug_list = sum(aug_data_checked['aug_data_cleaned'].tolist(), [])
print(f'flatten 후 총 문장 수: {len(aug_list)}')

In [ ]:
df_cleaned = pd.DataFrame(aug_list, columns=['aug_data_cleaned'])

# 중복 제거
df_cleaned = df_cleaned.drop_duplicates().reset_index(drop=True)
print(f'중복 제거 후: {len(df_cleaned)}개')

df_cleaned.to_csv(AUG_CLEANED_CSV, index=False, encoding='utf-8-sig')
print(f'저장 완료 → {AUG_CLEANED_CSV}')
df_cleaned.head()

## [4단계] 난독화 크롤링 (DPO 데이터셋용)

- **목적**: 난독화 사이트를 Selenium으로 자동화하여 DPO 학습용 증강 데이터 수집
- **방법**: [airbnbfy.hanmesoft.com](https://airbnbfy.hanmesoft.com/) 난독화 사이트 자동화
- **사용 용도**: LLM 증강 결과를 난독화 → `DPO_aug_pre_final1.csv` 생성 (현재 파일)
- **증강 전략**
  - 1차: `run_convert_with_range()` — 슬라이더 범위 지정해 체계적 순회
  - 2차: `run_convert_with_random()` — 랜덤 파라미터로 문장당 3회 반복
- **출력**: `input`(난독화 문장), `output`(원본 문장), `aug_type`(증강 방식) 컬럼의 CSV

> ⚠️ **3회 실행 필요**: 아래 `RUN_NUMBER`를 1 → 2 → 3으로 변경하며 총 3회 실행하세요.
> 결과 파일: `DPO_aug_pre_final1.csv`, `DPO_aug_pre_final2.csv`, `DPO_aug_pre_final3.csv`

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import random, time
import chromedriver_autoinstaller

In [ ]:
# ── 실행 번호 설정 (1, 2, 3 중 선택) ─────────────────────────────────────
RUN_NUMBER = 1  # ← 실행할 때마다 1 → 2 → 3으로 변경

CRAWL_OUTPUT_MAP = {1: FINAL_CSV_1, 2: FINAL_CSV_2, 3: FINAL_CSV_3}
CRAWL_OUTPUT_CSV = CRAWL_OUTPUT_MAP[RUN_NUMBER]
print(f'[RUN {RUN_NUMBER}] 출력 파일: {CRAWL_OUTPUT_CSV}')

In [ ]:
crawl_chrome_options = webdriver.ChromeOptions()
crawl_chrome_options.add_argument("lang=ko_KR")  # 한국어

# chromedriver_autoinstaller.install()

In [ ]:
aug_data_for_crawl = pd.read_csv(AUG_CLEANED_CSV, encoding="utf-8-sig")

In [ ]:
def convert_options_param(slider1_val, slider2_val, slider3_val, slider4_val):
    """난독화 사이트의 슬라이더 4개를 JS로 직접 조작하여 파라미터를 설정한다.

    Args:
        slider1_val: 소리나는 대로 연음 강도 (0~100)
        slider2_val: 뒤에 오는 자음을 받침으로 중복 강도 (0~100)
        slider3_val: 자모를 비슷한 발음으로 변환 강도 (0~100)
        slider4_val: 의미없는 받침 추가 강도 (0~100)
    """
    for slider_id, val in zip(
        ["formRange1", "formRange2", "formRange3", "formRange4"],
        [slider1_val, slider2_val, slider3_val, slider4_val],
    ):
        slider = crawl_driver.find_element(By.ID, slider_id)
        crawl_driver.execute_script("arguments[0].value = arguments[1];", slider, val)

In [ ]:
def run_convert_with_range(range1, range2, range3, range4):
    """[1차 증강] 슬라이더 범위를 지정하면 유효한 모든 조합을 순서대로 순회한다.

    특정 강도 구간의 조합을 빠짐없이 생성할 때 사용.
    예: range1=(50, 70) → 50, 60, 70 순서로 순회
    """
    valid_values = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

    range1_values = [v for v in valid_values if range1[0] <= v <= range1[1]]
    range2_values = [v for v in valid_values if range2[0] <= v <= range2[1]]
    range3_values = [v for v in valid_values if range3[0] <= v <= range3[1]]
    range4_values = [v for v in valid_values if range4[0] <= v <= range4[1]]

    for v1 in range1_values:
        for v2 in range2_values:
            for v3 in range3_values:
                for v4 in range4_values:
                    yield (v1, v2, v3, v4)


def run_convert_with_random():
    """[2차 증강] [50, 60, 70, 80, 90, 100] 중에서 4개 파라미터를 무작위로 선택한다.

    1차 증강 이후 추가 다양성 확보를 위해 사용.
    무한 generator로 next() 호출마다 새로운 조합을 반환.
    """
    valid_values = [50, 60, 70, 80, 90, 100]

    while True:
        yield tuple(random.choice(valid_values) for _ in range(4))


crawl_gen = run_convert_with_random()

In [ ]:
next(crawl_gen)

In [ ]:
crawl_rows = []  # 리스트에 모았다가 마지막에 한 번에 DataFrame 변환 (반복 append보다 빠름)

# 크롬창 실행
crawl_driver = webdriver.Chrome(options=crawl_chrome_options)
time.sleep(1)

# 한국 난독화 사이트 접속
url = "https://airbnbfy.hanmesoft.com/"
crawl_driver.get(url)
crawl_driver.implicitly_wait(3)

# UI 요소 객체 생성
text_box        = crawl_driver.find_element(By.XPATH, '//*[@id="root"]/div[1]/div/div[3]/textarea[1]')
convert         = crawl_driver.find_element(By.XPATH, '//*[@id="root"]/div[1]/div/div[3]/button[1]')
convert_options = crawl_driver.find_element(By.XPATH, '//*[@id="root"]/div[1]/div/div[3]/div/button[2]')

# 변환 옵션 패널 열기
convert_options.click()

for idx, sentence in enumerate(aug_data_for_crawl["aug_data_cleaned"][:3250]):
    try:
        text_box.clear()
        time.sleep(1)
        text_box.send_keys(sentence)
        time.sleep(1)

        for _ in range(3):  # 문장당 3회 반복 증강
            values = next(crawl_gen)
            print(f"[{idx+1}/3250] Executing with values: {values}")

            convert_options_param(*values)
            time.sleep(1)

            convert.click()

            converted_sentence = crawl_driver.find_element(
                By.XPATH, '//*[@id="root"]/div[1]/div/div[3]/textarea[2]'
            )

            crawl_rows.append({
                "input":    converted_sentence.text,
                "output":   sentence,
                "aug_type": "random",  # 1차 증강은 "range", 2차 증강은 "random"
            })

            time.sleep(1)

    except Exception as e:
        print(f"[ERROR] sentence {idx+1} 스킵 → {e}")
        continue

    # 100개마다 중간 저장 (크래시 시 데이터 손실 방지)
    if (idx + 1) % 100 == 0:
        pd.DataFrame(crawl_rows).to_csv(CRAWL_CKPT_CSV, index=False, encoding="utf-8-sig")
        print(f"[CHECKPOINT] {idx+1}번째 문장까지 저장 완료")

crawl_result = pd.DataFrame(crawl_rows)
crawl_result.to_csv(CRAWL_OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"\n완료: 총 {len(crawl_result)}개 저장 → {CRAWL_OUTPUT_CSV}")

In [ ]:
crawl_result

## [5단계] DPO 데이터셋 구축

- **목적**: SFT 모델의 추론 결과를 활용해 `(난독화 문장, 정답, 오답)` 3쌍의 DPO 학습 데이터셋 구성
- **추론 모델**: `Gyu96/llama3.1_additional_checkpoint1100` (SFT 최종 체크포인트)

## DPO 데이터셋 구성 전략

기존 SFT 학습 데이터에만 국한된 DPO 학습은 모델의 편향을 고착화하고 미학습 데이터에 대한 대응력을 낮출 위험이 있음.
따라서 **SFT 모델의 추론 실패 사례(Hard Negatives)**를 통해 취약점을 보완하는 동시에, LLM 기반 증강 데이터를 혼합하여
모델이 더 넓은 도메인에서 일반화된 선호도 판단 기준을 학습할 수 있도록 구성함.

| 데이터 | 역할 | 비고 |
|--------|------|------|
| train.csv + 난독화 증강 데이터 | SFT 학습 데이터 기반 오답 수집 | Hard Negatives 확보 |
| LLM 증강 → 난독화 데이터 | 미학습 데이터 기반 오답 수집 | 일반화 성능 향상 |

## 오답 필터링 기준

1. `output != inference` → 틀리게 예측한 행만
2. `inference` 중복 제거 → 동일한 오답 패턴 제거
3. `len(output) != len(inference)` → 길이까지 다른 확실한 오답만

In [ ]:
from vllm import LLM, SamplingParams

INFER_MODEL_NAME = 'Gyu96/llama3.1_additional_checkpoint1100'

vllm_infer_model = LLM(
    model=INFER_MODEL_NAME,
    tokenizer=INFER_MODEL_NAME,
    gpu_memory_utilization=0.9,
    max_model_len=8192
)

LLM_PROMPT = (
    "You are a helpful assistant specializing in restoring obfuscated Korean reviews. "
    "Your task is to transform the given obfuscated Korean review into a clear, correct, "
    "and natural-sounding Korean review that reflects its original meaning. "
    "Spacing and word length in the output must be restored to the same as in the input. "
    "Do not provide any description. Print only in Korean."
    "\n\n### Instruction:\n{}\n\n### Response:\n{}"
)


def batch_inference(df, batch_size=128):
    """배치 단위로 추론을 실행하고 결과 리스트를 반환한다.
    입력 문장 길이 기준으로 max_tokens를 동적 설정해 불필요한 생성을 방지.
    """
    results = []

    for i in tqdm(range(0, len(df), batch_size), desc='Inference'):
        batch = df['input'][i:i + batch_size].tolist()

        max_tokens = max(len(text) for text in batch)
        sampling_params = SamplingParams(
            temperature=0.05,
            top_p=0.95,
            max_tokens=max_tokens
        )

        prompts = [LLM_PROMPT.format(text, '') for text in batch]
        outputs = vllm_infer_model.generate(prompts, sampling_params)
        results.extend([o.outputs[0].text.strip() for o in outputs])

    return results

### 데이터 로드

- SFT 학습에 사용한 데이터 결합 + 중복 제거
- LLM 증강 → 난독화 데이터 별도 로드 (final1 + final2 + final3 결합)

In [ ]:
# SFT 학습 데이터 결합 + 중복 제거
train_sft = pd.read_csv(TRAIN_CSV,   encoding='utf-8-sig')
aug_texts  = pd.read_csv(AUG_CSV,    encoding='utf-8-sig')
aug_50     = pd.read_csv(AUG_CSV_50, encoding='utf-8-sig')
aug_100    = pd.read_csv(AUG_CSV_100, encoding='utf-8-sig')

sft_data = pd.concat([train_sft, aug_texts, aug_50, aug_100], ignore_index=True)
sft_data = sft_data.drop_duplicates().reset_index(drop=True)
print(f'SFT 데이터 크기: {len(sft_data)}')

# LLM 증강 → 난독화 데이터 로드 (3회 실행 결과 결합)
llm_aug_data = pd.concat(
    [
        pd.read_csv(FINAL_CSV_1, encoding='utf-8-sig'),
        pd.read_csv(FINAL_CSV_2, encoding='utf-8-sig'),
        pd.read_csv(FINAL_CSV_3, encoding='utf-8-sig'),
    ],
    ignore_index=True,
)
llm_aug_data = llm_aug_data.drop_duplicates().dropna().reset_index(drop=True)
print(f'LLM 증강 데이터 크기: {len(llm_aug_data)}')

### SFT 학습 데이터 추론

In [ ]:
sft_inferences = batch_inference(sft_data)

sft_data = sft_data.drop(columns='ID', errors='ignore')
sft_data['inference'] = sft_inferences
sft_data.to_csv(OUTPUT_SFT_CSV, index=False, encoding='utf-8-sig')
print(f'SFT 추론 완료 → {OUTPUT_SFT_CSV}')

### LLM 증강 데이터 추론

In [ ]:
llm_aug_inferences = batch_inference(llm_aug_data)

llm_aug_data['inference'] = pd.Series(llm_aug_inferences).reindex(llm_aug_data.index)
llm_aug_data.to_csv(OUTPUT_AUG_CSV, index=False, encoding='utf-8-sig')
print(f'LLM 증강 데이터 추론 완료 → {OUTPUT_AUG_CSV}')

### 오답 필터링 (Hard Negatives 추출)

두 데이터 모두 동일한 3단계 필터링 적용:
1. `output != inference` → 틀리게 예측한 행만
2. `inference` 중복 제거
3. `len(output) != len(inference)` → 길이까지 다른 확실한 오답만

In [ ]:
def filter_hard_negatives(df):
    """추론 실패 샘플(Hard Negatives)만 추출한다.

    1단계: output != inference (틀린 예측)
    2단계: inference 중복 제거
    3단계: output과 inference 길이가 다른 확실한 오답만
    """
    # 1단계: 틀린 예측만
    df_wrong = df[df["output"].str.strip() != df["inference"].str.strip()]

    # 2단계: inference 중복 제거
    df_dedup = df_wrong.drop_duplicates(subset=["inference"], keep="first")

    # 3단계: 길이까지 다른 확실한 오답만
    df_filtered = df_dedup[
        df_dedup["output"].str.len() != df_dedup["inference"].str.len()
    ]

    return df_filtered


sft_hard_neg = filter_hard_negatives(sft_data)
llm_hard_neg = filter_hard_negatives(llm_aug_data)

print(f"SFT 데이터 오답: {len(sft_hard_neg)}개")
print(f"LLM 증강 데이터 오답: {len(llm_hard_neg)}개")

### 최종 DPO 데이터셋 구성

- SFT 오답 + LLM 증강 오답 결합
- `inference`의 NaN은 `input` 값으로 채운 뒤 나머지 NaN 제거
- 최종 컬럼: `input`(난독화 문장), `output`(정답), `inference`(오답)

In [ ]:
# SFT 오답 + LLM 증강 오답 결합
dpo_dataset = pd.concat([sft_hard_neg, llm_hard_neg], ignore_index=True)

# inference NaN → input 값으로 채우기 (output 기준 첫 번째 행만)
mask = dpo_dataset['inference'].isnull() & ~dpo_dataset.duplicated(
    subset=['output'], keep='first'
)
dpo_dataset.loc[mask, 'inference'] = dpo_dataset.loc[mask, 'input']

# 나머지 NaN 제거
dpo_dataset = dpo_dataset.dropna(subset=['inference']).reset_index(drop=True)

print(f'최종 DPO 데이터셋 크기: {len(dpo_dataset)}')
dpo_dataset.to_csv(OUTPUT_FINAL_CSV, index=False, encoding='utf-8-sig')
print(f'저장 완료 → {OUTPUT_FINAL_CSV}')
dpo_dataset.head(3)